In [4]:
# ===============================
# evaluate_roc_auc.py
# (PER-CLASS + OVERALL + ROC CURVE)
# ===============================

import torch
import numpy as np
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report,
    roc_auc_score,
    roc_curve,
    auc
)

from sklearn.preprocessing import label_binarize

from data_preprocessing import load_and_preprocess_data
from tokenizer import BiologicalTokenizer
from model import MiRNADataset, MiRNADiseaseTransformer


# =========================================================
# LOG FILE SETUP
# =========================================================
os.makedirs("logs", exist_ok=True)

log_path = "logs/evaluation_log_roc_auc-100.txt"
log_file = open(log_path, "w")

def log(text):
    print(text)
    log_file.write(text + "\n")

log("==== MODEL EVALUATION STARTED ====")


# =========================================================
# Collate Function
# =========================================================
def collate_fn(batch):

    max_len = max(len(item['sequence_ids']) for item in batch)

    seqs, masks = [], []
    mirnas, genes = [], []
    disease_labels, context_labels = [], []

    for item in batch:

        pad_len = max_len - len(item['sequence_ids'])

        seqs.append(torch.cat([
            item['sequence_ids'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))

        masks.append(torch.cat([
            item['attention_mask'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))

        mirnas.append(item['mirna_ids'])
        genes.append(item['gene_ids'])

        disease_labels.append(item['disease_labels'])
        context_labels.append(item['context_labels'])

    return {
        'sequence_ids': torch.stack(seqs),
        'attention_mask': torch.stack(masks),
        'mirna_ids': torch.stack(mirnas),
        'gene_ids': torch.stack(genes),
        'disease_labels': torch.stack(disease_labels),
        'context_labels': torch.stack(context_labels)
    }


# =========================================================
# Load Data
# =========================================================
train, val, test, dis_enc, ctx_enc = load_and_preprocess_data(
    "data/train_dataset.csv",
    min_samples=5
)

log(f"Test samples: {len(test)}")
log(f"Disease classes: {len(dis_enc.classes_)}")
log(f"Context classes: {len(ctx_enc.classes_)}")


# =========================================================
# Load Tokenizer
# =========================================================
tok = BiologicalTokenizer.load("saved_models/tokenizer.pth")

log("Tokenizer loaded.")


# =========================================================
# Load Model
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiRNADiseaseTransformer(
    vocab_size=len(tok.char_to_idx),
    mirna_vocab=len(tok.mirna_char_to_idx),
    gene_vocab=len(tok.gene_to_idx),
    num_disease=len(dis_enc.classes_),
    num_context=len(ctx_enc.classes_)
)

model.load_state_dict(torch.load("saved_models/model.pth", map_location=device))

model.to(device)
model.eval()

log(f"Model loaded on device: {device}")


# =========================================================
# Test Loader
# =========================================================
test_loader = DataLoader(
    MiRNADataset(test, tok),
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)


# =========================================================
# Evaluation Loop
# =========================================================
all_probs_d, all_labels_d = [], []
all_probs_c, all_labels_c = [], []

with torch.no_grad():

    for batch in test_loader:

        seq_ids = batch['sequence_ids'].to(device)
        att_mask = batch['attention_mask'].to(device)
        mirna_ids = batch['mirna_ids'].to(device)
        gene_ids = batch['gene_ids'].to(device)

        ctx_labels = batch['context_labels'].to(device)
        disease_labels = batch['disease_labels'].to(device)

        disease_logits, context_logits = model(
            seq_ids, att_mask, mirna_ids, gene_ids, ctx_labels
        )

        probs_d = torch.softmax(disease_logits, dim=1)

        all_probs_d.append(probs_d.cpu().numpy())
        all_labels_d.append(disease_labels.cpu().numpy())

        probs_c = torch.softmax(context_logits, dim=1)[:, 1]

        all_probs_c.append(probs_c.cpu().numpy())
        all_labels_c.append(ctx_labels.cpu().numpy())


all_probs_d = np.concatenate(all_probs_d, axis=0)
all_labels_d = np.concatenate(all_labels_d, axis=0)

all_probs_c = np.concatenate(all_probs_c, axis=0)
all_labels_c = np.concatenate(all_labels_c, axis=0)


# =========================================================
# Predictions
# =========================================================
preds_d = np.argmax(all_probs_d, axis=1)
preds_c = (all_probs_c >= 0.5).astype(int)


# =========================================================
# DISEASE METRICS
# =========================================================
log("\n==============================")
log("DISEASE RESULTS")
log("==============================")

acc_d = accuracy_score(all_labels_d, preds_d)

report_d = classification_report(
    all_labels_d,
    preds_d,
    target_names=dis_enc.classes_,
    output_dict=True,
    zero_division=0
)

for cls in dis_enc.classes_:

    m = report_d[cls]

    log(f"{cls:20} - Precision: {m['precision']:.4f}, "
        f"Recall: {m['recall']:.4f}, "
        f"F1-score: {m['f1-score']:.4f}")


macro_d = report_d["macro avg"]
weighted_d = report_d["weighted avg"]

log("\n------ Overall Disease Metrics ------")

log(f"Accuracy : {acc_d:.4f}")
log(f"Precision : {weighted_d['precision']:.4f}")
log(f"Recall : {weighted_d['recall']:.4f}")
log(f"F1-score : {weighted_d['f1-score']:.4f}")


roc_auc_d = roc_auc_score(all_labels_d, all_probs_d, multi_class="ovr")

log(f"ROC-AUC : {roc_auc_d:.4f}")


# =========================================================
# CONTEXT METRICS
# =========================================================
log("\n==============================")
log("CONTEXT RESULTS")
log("==============================")

acc_c = accuracy_score(all_labels_c, preds_c)

report_c = classification_report(
    all_labels_c,
    preds_c,
    target_names=ctx_enc.classes_,
    output_dict=True,
    zero_division=0
)

for cls in ctx_enc.classes_:

    m = report_c[cls]

    log(f"{cls:20} - Precision: {m['precision']:.4f}, "
        f"Recall: {m['recall']:.4f}, "
        f"F1-score: {m['f1-score']:.4f}")


macro_c = report_c["macro avg"]
weighted_c = report_c["weighted avg"]

log("\n------ Overall Context Metrics ------")

log(f"Accuracy : {acc_c:.4f}")
log(f"Precision : {weighted_c['precision']:.4f}")
log(f"Recall : {weighted_c['recall']:.4f}")
log(f"F1-score : {weighted_c['f1-score']:.4f}")


roc_auc_c = roc_auc_score(all_labels_c, all_probs_c)

log(f"ROC-AUC : {roc_auc_c:.4f}")


# =========================================================
# ROC CURVE PLOTS
# =========================================================
os.makedirs("figures", exist_ok=True)

n_classes = len(dis_enc.classes_)

y_bin = label_binarize(all_labels_d, classes=np.arange(n_classes))

plt.figure(figsize=(8,6))

for i in range(n_classes):

    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs_d[:, i])
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr,
             label=f"{dis_enc.classes_[i]} (AUC={roc_auc:.2f})")

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("Disease ROC Curve")

plt.legend()

plt.tight_layout()

plt.savefig("figures/disease_roc_curve-100.png", dpi=300)

plt.close()


fpr_c, tpr_c, _ = roc_curve(all_labels_c, all_probs_c)
roc_auc_context = auc(fpr_c, tpr_c)

plt.figure(figsize=(6,5))

plt.plot(fpr_c, tpr_c, label=f"AUC={roc_auc_context:.2f}")

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("Context ROC Curve")

plt.legend()

plt.tight_layout()

plt.savefig("figures/context_roc_curve-100.png", dpi=300)

plt.close()


# =========================================================
# CONFUSION MATRIX
# =========================================================
cm_d = confusion_matrix(all_labels_d, preds_d)
cm_c = confusion_matrix(all_labels_c, preds_c)

plt.figure(figsize=(10,6))

sns.heatmap(cm_d, annot=True, fmt="d",
            xticklabels=dis_enc.classes_,
            yticklabels=dis_enc.classes_)

plt.title("Disease Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.tight_layout()

plt.savefig("figures/disease_confusion_matrix-100.png", dpi=300)

plt.close()


plt.figure(figsize=(6,5))

sns.heatmap(cm_c, annot=True, fmt="d",
            xticklabels=ctx_enc.classes_,
            yticklabels=ctx_enc.classes_)

plt.title("Context Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.tight_layout()

plt.savefig("figures/context_confusion_matrix-100.png", dpi=300)

plt.close()


log("Figures saved in figures folder")


# =========================================================
# FINISH
# =========================================================
log("\n==== EVALUATION COMPLETED SUCCESSFULLY ====")

log_file.close()

print(f"\nEvaluation log saved at: {log_path}")

==== MODEL EVALUATION STARTED ====
Test samples: 222
Disease classes: 15
Context classes: 2
Tokenizer loaded.
Model loaded on device: cpu


C:\Users\Nabilah\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(



DISEASE RESULTS
Amyotrophic Lateral Sclerosis - Precision: 0.7500, Recall: 0.7500, F1-score: 0.7500
Brain Neoplasms      - Precision: 0.3333, Recall: 1.0000, F1-score: 0.5000
Cholangiocarcinoma   - Precision: 0.5000, Recall: 0.5714, F1-score: 0.5333
Esophageal Neoplasms - Precision: 0.6667, Recall: 0.5000, F1-score: 0.5714
Glioblastoma         - Precision: 0.7500, Recall: 0.3750, F1-score: 0.5000
Glioma               - Precision: 0.7000, Recall: 0.8750, F1-score: 0.7778
Hematologic Neoplasms - Precision: 0.3333, Recall: 0.3333, F1-score: 0.3333
Leukemia             - Precision: 0.8214, Recall: 0.6571, F1-score: 0.7302
Lung Neoplasms       - Precision: 0.8043, Recall: 0.8409, F1-score: 0.8222
Lymphoma             - Precision: 0.6757, Recall: 0.7143, F1-score: 0.6944
Medulloblastoma      - Precision: 0.7143, Recall: 0.4545, F1-score: 0.5556
Multiple Myeloma     - Precision: 0.7500, Recall: 1.0000, F1-score: 0.8571
Neuroblastoma        - Precision: 0.3846, Recall: 0.8333, F1-score: 0.526